In [1]:
import pickle
import cv2
import mediapipe as mp
from collections import deque
import time
import numpy as np
from tensorflow.keras.models import load_model

import pyttsx3
import threading
import queue


_tts_queue = queue.Queue()

def _tts_worker():
    engine = pyttsx3.init()
    engine.setProperty('rate', 150)
    engine.setProperty('volume', 1.0)
    while True:
        text = _tts_queue.get()          
        if text is None:                
            break
        engine.say(text)
        engine.runAndWait()

_tts_thread = threading.Thread(target=_tts_worker, daemon=True)
_tts_thread.start()

def speak(text):
   
    while not _tts_queue.empty():
        try:
            _tts_queue.get_nowait()
        except queue.Empty:
            break
    _tts_queue.put(text)

static_model = pickle.load(open('Static-Model-Sentence/model.p', 'rb'))['model']
motion_model = load_model('Motion-LSTM-Model/motion_model.h5')

motion_data = pickle.load(open('Motion-LSTM-Model/motion_data.pickle', 'rb'))
actions = motion_data['actions']

STATIC_HELP_IMG_PATH = r'C:\Users\Prompt\Downloads\Sign-Language-Recognition-main\Sign-Language-Recognition-main\help\static_help.jpg'   # <- your static-help image path
MOTION_HELP_IMG_PATH = r'C:\Users\Prompt\Downloads\Sign-Language-Recognition-main\Sign-Language-Recognition-main\help\motion_help.png'   # <- your motion-help image path

_static_help_img = cv2.imread(STATIC_HELP_IMG_PATH)
_motion_help_img = cv2.imread(MOTION_HELP_IMG_PATH)

HELP_WIN_STATIC = 'Static Help  [Press 1 to close]'
HELP_WIN_MOTION = 'Motion Help  [Press 2 to close]'
HELP_WIN_W, HELP_WIN_H = 480, 360   
HELP_WIN_X, HELP_WIN_Y = 30, 30     

def _resize_help(img):

    if img is None:
        placeholder = np.zeros((HELP_WIN_H, HELP_WIN_W, 3), dtype=np.uint8)
        cv2.putText(placeholder, 'Image not found', (60, HELP_WIN_H // 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
        return placeholder
    return cv2.resize(img, (HELP_WIN_W, HELP_WIN_H))

static_help_display = _resize_help(_static_help_img)
motion_help_display = _resize_help(_motion_help_img)

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    min_detection_confidence=0.5,
    max_num_hands=1
)

def extract_hand_features(hand_landmarks):
    x_, y_ = [], []
    for lm in hand_landmarks.landmark:
        x_.append(lm.x)
        y_.append(lm.y)

    data_aux = []
    for lm in hand_landmarks.landmark:
        data_aux.append(lm.x - min(x_))
        data_aux.append(lm.y - min(y_))

    return data_aux


def draw_multiline_text(frame, text, x, y, max_width, line_height=35):
    words = text.split(' ')
    lines = []
    current_line = ""

    for word in words:
        test_line = current_line + word + " "
        text_size = cv2.getTextSize(test_line, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0]

        if text_size[0] > max_width:
            lines.append(current_line)
            current_line = word + " "
        else:
            current_line = test_line

    lines.append(current_line)

    for i, line in enumerate(lines):
        cv2.putText(frame, line.strip(), (x, y + i * line_height),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)


buffer = deque(maxlen=10)
sentence = ""
prev_char = ""
last_added_time = 0
cooldown = 4.0

sequence = []

mode = "static"

show_static_help = False
show_motion_help = False

cap = cv2.VideoCapture(0)

cv2.namedWindow('Hybrid ISL System', cv2.WND_PROP_FULLSCREEN)
cv2.setWindowProperty('Hybrid ISL System', cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    frame = cv2.flip(frame, 1)

    h, w, _ = frame.shape
    panel_width = 500

    panel = np.zeros((h, panel_width, 3), dtype=np.uint8)

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    display_char = ""
    current_time = time.time()

    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]
        features = extract_hand_features(hand_landmarks)

        mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        if mode == "motion":
            sequence.append(features)
            sequence = sequence[-30:]

            if len(sequence) == 30:
                pred = motion_model.predict(np.expand_dims(sequence, axis=0))[0]
                action = actions[np.argmax(pred)]
                display_char = action

                if (current_time - last_added_time) > cooldown:
                    sentence += action + " "
                    last_added_time = current_time
                    prev_char = ""
                    speak(action)

        else:
            prediction = static_model.predict([features])[0]
            buffer.append(prediction)

            if len(buffer) == buffer.maxlen:
                stable_char = max(set(buffer), key=buffer.count)
                display_char = stable_char

                if stable_char != prev_char and (current_time - last_added_time) > cooldown:
                    if stable_char == "space":
                        sentence += " "
                    else:
                        sentence += stable_char

                    prev_char = stable_char
                    last_added_time = current_time

    if len(sentence) > 120:
        sentence = sentence[-120:]

    cv2.putText(panel, f'Mode: {mode.upper()}',
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 255, 0), 2)

    cv2.putText(panel, f'Current:',
                (20, 100), cv2.FONT_HERSHEY_SIMPLEX,
                0.9, (0, 255, 255), 2)

    cv2.putText(panel, display_char,
                (20, 140), cv2.FONT_HERSHEY_SIMPLEX,
                1.2, (255, 255, 255), 3)

    draw_multiline_text(panel, f'Sentence: {sentence}', 20, 220, 450)

    cv2.putText(panel, '1=Static Help | 2=Motion Help',
                (20, h - 155), cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (180, 180, 0), 1)

    draw_multiline_text(
        panel,
        'M=Motion | N=Static | S=Speak | Q=Quit | C=Clear | B=Backspace | W=Delete Word',
        20, h - 120, 450, 30
    )

    combined = np.hstack((frame, panel))

    cv2.imshow('Hybrid ISL System', combined)

    if show_static_help:
        cv2.imshow(HELP_WIN_STATIC, static_help_display)
        cv2.moveWindow(HELP_WIN_STATIC, HELP_WIN_X, HELP_WIN_Y)

    if show_motion_help:
        cv2.imshow(HELP_WIN_MOTION, motion_help_display)
        cv2.moveWindow(HELP_WIN_MOTION, HELP_WIN_X, HELP_WIN_Y)
 
    key = cv2.waitKey(1) & 0xFF

    if key == ord('m'):
        mode = "motion"
        sequence = []

    elif key == ord('n'):
        mode = "static"
        buffer.clear()


    elif key == ord('q'):
        break

    elif key == ord('c'):
        sentence = ""
        prev_char = ""

    elif key == ord('b'):
        sentence = sentence[:-1]

    elif key == ord('w'):
        sentence = sentence.rstrip()
        sentence = " ".join(sentence.split(" ")[:-1])

    elif key == ord('s'):           
        text = sentence.strip()
        if text:
            speak(text)

    elif key == ord('1'):       
        show_static_help = not show_static_help
        if not show_static_help:
            cv2.destroyWindow(HELP_WIN_STATIC)

    elif key == ord('2'):       
        show_motion_help = not show_motion_help
        if not show_motion_help:
            cv2.destroyWindow(HELP_WIN_MOTION)

cap.release()
cv2.destroyAllWindows()

In [2]:
static_model = pickle.load(open('Static-Model-Sentence/model.p', 'rb'))['model']
motion_model = load_model('Motion-LSTM-Model/motion_model.h5')

motion_data = pickle.load(open('Motion-LSTM-Model/motion_data.pickle', 'rb'))